# IBM Predictive Maintenance - Interview Walkthrough

This notebook is structured for live interview delivery:

1. Business framing
2. Quick telemetry EDA
3. Model quality and thresholding
4. Operational alert strategy
5. Business recommendations

In [ ]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

project_root = Path.cwd().resolve().parent
artifacts_dir = project_root / "artifacts"
processed_data_path = project_root / "data" / "processed" / "synthetic_maintenance.csv"
metrics_path = artifacts_dir / "metrics.json"
feature_importance_path = artifacts_dir / "feature_importance.csv"
scored_path = artifacts_dir / "scored_test.csv"
alerts_path = artifacts_dir / "alerts.csv"

print("Project root:", project_root)

In [ ]:
# Ensure artifacts exist so notebook is runnable end-to-end.
if not metrics_path.exists() or not scored_path.exists():
    print("Artifacts not found. Running training script...")
    subprocess.run(["python3", str(project_root / "src" / "train.py")], check=True)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

raw_data = pd.read_csv(processed_data_path)
scored = pd.read_csv(scored_path)
alerts = pd.read_csv(alerts_path)
feature_importance = pd.read_csv(feature_importance_path)

print("Rows in telemetry dataset:", len(raw_data))
print("Rows in holdout scored set:", len(scored))
print("Current alert count:", len(alerts))
print("Selected model:", metrics["selected_model"])

## 1) Problem Framing

Goal: flag assets with elevated failure risk in the next 7 days so operations can intervene early.

Primary interview message:

- This is not just model accuracy.
- This is a decision system balancing missed failures vs alert fatigue.

In [ ]:
failure_rate = raw_data["FailureNext7Days"].mean()
print(f"Overall failure-next-7-days rate: {failure_rate:.2%}")

for col in ["EquipmentType", "OperatingMode", "Site"]:
    summary = (
        raw_data.groupby(col, dropna=False)["FailureNext7Days"]
        .agg(["count", "mean"])
        .rename(columns={"count": "n", "mean": "failure_rate"})
        .sort_values("failure_rate", ascending=False)
    )
    print(f"\nFailure rate by {col}:")
    display(summary)

corr_cols = [
    "FailureNext7Days",
    "SensorVibration",
    "SensorTemp",
    "SensorPressure",
    "DaysSinceMaintenance",
    "AssetAgeYears",
    "OperatingHours",
]
corr = raw_data[corr_cols].corr(numeric_only=True)
print("\nCorrelation snapshot to target:")
display(corr[["FailureNext7Days"]].sort_values("FailureNext7Days", ascending=False))

## 2) Model Performance

The training script compares logistic regression and random forest with CV ROC-AUC, then evaluates holdout performance.

For predictive maintenance, I focus on:

- Ranking quality (ROC-AUC, PR-AUC)
- Operational quality at chosen threshold (precision + recall)

In [ ]:
print("Cross-validated ROC-AUC by model:")
for model_name, auc in metrics["cv_roc_auc"].items():
    print(f"  {model_name}: {auc:.3f}")

print("\nSelected model:", metrics["selected_model"])
print("Decision threshold:", metrics["decision_threshold"])

print("\nHoldout test metrics:")
for metric_name, value in metrics["test_metrics"].items():
    print(f"  {metric_name}: {value:.3f}")

print("\nSubgroup checks:")
for group_name, group_values in metrics["subgroup_metrics"].items():
    print(f"\n{group_name}")
    if not group_values:
        print("  (no eligible segments)")
        continue
    display(pd.DataFrame(group_values).T.sort_values("n_samples", ascending=False))

## 3) Threshold Strategy and Alert Operations

A good interview answer includes how to tune threshold based on field capacity.

If maintenance teams can only process a limited number of alerts per week,
choose a threshold that maximizes recall under that capacity constraint.

In [ ]:
y_true = scored["actual_failure"].astype(int)
y_prob = scored["failure_probability"].astype(float)

rows = []
for threshold in np.arange(0.20, 0.91, 0.05):
    y_pred = (y_prob >= threshold).astype(int)
    rows.append(
        {
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "alert_rate": float(y_pred.mean()),
            "n_alerts": int(y_pred.sum()),
        }
    )

threshold_table = pd.DataFrame(rows)
display(
    threshold_table.style.format(
        {
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "f1": "{:.3f}",
            "alert_rate": "{:.3f}",
        }
    )
)

print("Top 10 highest-risk alerts:")
display(alerts.sort_values("failure_probability", ascending=False).head(10))

In [ ]:
print("Top 15 model drivers:")
display(feature_importance.head(15))

## 4) IBM-Style Recommendations

- Run daily batch scoring over active assets.
- Send top-risk alerts into maintenance dispatch tooling.
- Track weekly precision/recall and alert volume against capacity.
- Monitor subgroup performance by equipment type and site.
- Retrain quarterly or when drift and KPI degradation are detected.

## Interview Close

"I built this as a decision pipeline, not just a model: realistic split strategy, threshold governance, and operations-ready alert outputs tied to business KPIs like downtime and maintenance efficiency."